In [1]:
#问题3：编写代码，使用默认参数，实现逻辑回归、决策树、k近邻和支持向量机分类器，计算各个分类器对测试集合的预测精度。

In [2]:
# 公共初始化代码｜依赖：第一题已保存的数据；输出：环境与本题输出目录
from pathlib import Path
from datetime import datetime
import hashlib
import json
import sys
import uuid
import warnings
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

# 自动定位仓库中的作业目录，支持从仓库根目录或notebooks启动。
_location = Path.cwd()
for _parent in (_location, *_location.parents):
    _notebooks = _parent if _parent.name == "notebooks" else _parent / "notebooks"
    if (_notebooks / "imbalanced_classification" / "common.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
else:
    raise FileNotFoundError("未找到作业目录，请从仓库根目录或notebooks启动")
from imbalanced_classification.common import (
    PROJECT_DIR, check_environment, SOURCE_DIR, load_fixed_dataset, dataset_sha256
)
check_environment()
source_manifest = json.loads((SOURCE_DIR / "results/manifest.json").read_text(encoding="utf-8"))
assert sklearn.__version__ == source_manifest["versions"]["scikit-learn"], "sklearn版本与第一题不同，请先核对环境"
saved = load_fixed_dataset(SOURCE_DIR)
X, y = saved["X"], saved["y"]
train_idx, test_idx = saved["train_idx"], saved["test_idx"]
data_hash = dataset_sha256(X, y, train_idx, test_idx)
assert data_hash == source_manifest["data_split_sha256"], "第一题数据或划分发生变化"
X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
LABELS, MODEL_SEED = np.arange(10), source_manifest["model_seed"]
assert np.array_equal(np.unique(y_train), LABELS) and np.array_equal(np.unique(y_test), LABELS)
Q3_DIR = PROJECT_DIR / "q3_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
Q3_DIR.mkdir(parents=True, exist_ok=False)
print("问题3 v1.0；Python", sys.version.split()[0], "；scikit-learn", sklearn.__version__)
print("数据：", X_train.shape, X_test.shape, "；模型种子：", MODEL_SEED)
print("本题输出目录：", Q3_DIR)

问题3 v1.0；Python 3.12.13 ；scikit-learn 1.9.0
数据： (3600, 24) (1200, 24) ；模型种子： 20261022
本题输出目录： D:\Python for Data Analysis\notebooks\imbalanced_classification\q3_runs\20261021_111602_87b2fd


In [3]:
# 四个模型的实际完整配置

models = {
    "LogisticRegression": LogisticRegression(random_state=MODEL_SEED),
    "DecisionTree": DecisionTreeClassifier(random_state=MODEL_SEED),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(random_state=MODEL_SEED),
}
model_parameters = {name: model.get_params(deep=False) for name, model in models.items()}
for name, parameters in model_parameters.items():
    print(name)
    print(json.dumps(parameters, ensure_ascii=False, sort_keys=True))
print("以上除random_state外均为默认参数；本题不做标准化。")

LogisticRegression
{"C": 1.0, "class_weight": null, "dual": false, "fit_intercept": true, "intercept_scaling": 1, "l1_ratio": 0.0, "max_iter": 100, "n_jobs": null, "penalty": "deprecated", "random_state": 20261022, "solver": "lbfgs", "tol": 0.0001, "verbose": 0, "warm_start": false}
DecisionTree
{"ccp_alpha": 0.0, "class_weight": null, "criterion": "gini", "max_depth": null, "max_features": null, "max_leaf_nodes": null, "min_impurity_decrease": 0.0, "min_samples_leaf": 1, "min_samples_split": 2, "min_weight_fraction_leaf": 0.0, "monotonic_cst": null, "random_state": 20261022, "splitter": "best"}
KNN
{"algorithm": "auto", "leaf_size": 30, "metric": "minkowski", "metric_params": null, "n_jobs": null, "n_neighbors": 5, "p": 2, "weights": "uniform"}
SVC
{"C": 1.0, "break_ties": false, "cache_size": 200, "class_weight": null, "coef0": 0.0, "decision_function_shape": "ovr", "degree": 3, "gamma": "scale", "kernel": "rbf", "max_iter": -1, "probability": "deprecated", "random_state": 20261022, 

In [4]:
# 依赖：模型配置；输出：训练状态、预测和警告
predictions, warning_rows, fit_diagnostics = {}, [], {}
for name, model in models.items():
    print(f"开始训练：{name}", flush=True)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")  # 记录并在下方完整输出，不忽略收敛问题。
        model.fit(X_train, y_train)
        predictions[name] = model.predict(X_test)
    for item in caught:
        warning_rows.append({"model": name, "category": item.category.__name__,
                             "message": str(item.message),
                             "is_convergence_warning": issubclass(item.category, ConvergenceWarning)})
        print(f"警告 [{item.category.__name__}]：{item.message}")
    fit_diagnostics[name] = {
        "n_iter": np.asarray(model.n_iter_).tolist() if hasattr(model, "n_iter_") else None,
        "fit_status": int(model.fit_status_) if hasattr(model, "fit_status_") else None,
    }
    print(f"{name}：预测{len(predictions[name])}例；捕获警告{len(caught)}条")
    print("迭代/求解状态：", fit_diagnostics[name])

开始训练：LogisticRegression
LogisticRegression：预测1200例；捕获警告0条
迭代/求解状态： {'n_iter': [82], 'fit_status': None}
开始训练：DecisionTree
DecisionTree：预测1200例；捕获警告0条
迭代/求解状态： {'n_iter': None, 'fit_status': None}
开始训练：KNN
KNN：预测1200例；捕获警告0条
迭代/求解状态： {'n_iter': None, 'fit_status': None}
开始训练：SVC
SVC：预测1200例；捕获警告0条
迭代/求解状态： {'n_iter': [943, 929, 652, 769, 781, 780, 648, 605, 434, 786, 556, 693, 537, 647, 489, 424, 399, 620, 701, 546, 521, 495, 425, 473, 428, 381, 523, 382, 422, 289, 408, 422, 316, 339, 314, 444, 372, 361, 320, 337, 288, 252, 223, 274, 186], 'fit_status': 0}


In [5]:
# 依赖：预测；输出：测试集Accuracy、macro-F1及逐类指标
summary_rows, class_parts = [], []
for name, y_pred in predictions.items():
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, labels=LABELS, average="macro", zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, labels=LABELS, average=None, zero_division=0
    )
    summary_rows.append({"model": name, "accuracy": acc, "macro_f1": macro_f1})
    class_parts.append(pd.DataFrame({
        "model": name, "class_label": LABELS, "support": support,
        "precision": precision, "recall": recall, "f1": f1,
    }))
summary = pd.DataFrame(summary_rows)
summary["accuracy_rank"] = summary["accuracy"].rank(ascending=False, method="min").astype(int)
summary["macro_f1_rank"] = summary["macro_f1"].rank(ascending=False, method="min").astype(int)
per_class = pd.concat(class_parts, ignore_index=True)
print("测试集总体指标（排名仅描述本次结果，不用测试集调参）：")
display(summary.round(6))
print("测试集逐类F1（标签0—9全部保留）：")
display(per_class.pivot(index="class_label", columns="model", values="f1").round(6))
print("测试集逐类Recall：")
display(per_class.pivot(index="class_label", columns="model", values="recall").round(6))

测试集总体指标（排名仅描述本次结果，不用测试集调参）：


,model,accuracy,macro_f1,accuracy_rank,macro_f1_rank
0,LogisticRegression,0.565833,0.386783,3,3
1,DecisionTree,0.495833,0.359136,4,4
2,KNN,0.728333,0.598193,2,2
3,SVC,0.795000,0.669896,1,1


测试集逐类F1（标签0—9全部保留）：


model,DecisionTree,KNN,LogisticRegression,SVC
class_label,,,,
0,0.690122,0.819026,0.756184,0.879012
1,0.585683,0.770751,0.592742,0.844262
2,0.436137,0.733766,0.364217,0.790419
3,0.484848,0.776786,0.746032,0.863071
4,0.285714,0.600000,0.204082,0.674699
5,0.335570,0.450450,0.333333,0.622951
6,0.236220,0.534884,0.088235,0.567901
7,0.210526,0.468750,0.305085,0.483871
8,0.213333,0.370370,0.363636,0.551724


测试集逐类Recall：


model,DecisionTree,KNN,LogisticRegression,SVC
class_label,,,,
0,0.672823,0.931398,0.846966,0.939314
1,0.600000,0.866667,0.653333,0.915556
2,0.451613,0.729032,0.367742,0.851613
3,0.421053,0.763158,0.824561,0.912281
4,0.285714,0.494505,0.164835,0.615385
5,0.337838,0.337838,0.283784,0.513514
6,0.267857,0.410714,0.053571,0.410714
7,0.232558,0.348837,0.209302,0.348837
8,0.216216,0.270270,0.270270,0.432432


In [6]:
# 辅助代码｜输出：本题结果文件、警告计数和保存路径
summary.to_csv(Q3_DIR / "q3_summary.csv", index=False, encoding="utf-8-sig")
per_class.to_csv(Q3_DIR / "q3_per_class.csv", index=False, encoding="utf-8-sig")
prediction_table = pd.DataFrame({"sample_id": test_idx, "y_true": y_test, **predictions})
prediction_table.to_csv(Q3_DIR / "q3_predictions.csv", index=False, encoding="utf-8-sig")
confusion_rows = []
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=LABELS)
    assert cm.sum() == len(y_test)
    confusion_rows.extend({"model": name, "true_label": int(i), "predicted_label": int(j),
                           "count": int(cm[i, j])} for i in LABELS for j in LABELS)
pd.DataFrame(confusion_rows).to_csv(Q3_DIR / "q3_confusion_counts.csv", index=False, encoding="utf-8-sig")
for filename, content in (("q3_model_parameters.json", model_parameters),
                          ("q3_warnings.json", warning_rows), ("q3_fit_diagnostics.json", fit_diagnostics)):
    (Q3_DIR / filename).write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding="utf-8")
record = dict(code_version="1.0", source_dir=str(SOURCE_DIR), data_split_sha256=data_hash,
              python=sys.version, sklearn=sklearn.__version__, executable=sys.executable,
              model_seed=MODEL_SEED, models=list(models), preprocessing="none",
              labels=LABELS.tolist(), zero_division=0, train_samples=len(y_train), test_samples=len(y_test),
              warning_count=len(warning_rows), created_at=datetime.now().astimezone().isoformat(),
              confusion_matrix="raw counts; rows=true labels, columns=predicted labels",
              output_dir=str(Q3_DIR))
(Q3_DIR / "q3_manifest.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存完成：", Q3_DIR)
print("总警告数：", len(warning_rows), "；收敛警告数：", sum(x["is_convergence_warning"] for x in warning_rows))
print("请发送核心代码和运行输出截图；有警告也保留，先不要修改默认参数。")

保存完成： D:\Python for Data Analysis\notebooks\imbalanced_classification\q3_runs\20261021_111602_87b2fd
总警告数： 0 ；收敛警告数： 0
请发送核心代码和运行输出截图；有警告也保留，先不要修改默认参数。
